# NOAA Environmental Data Builder (Colab)

This notebook pulls **hourly environmental variables** needed for thermal normalization and exports a clean CSV.

**Outputs (hourly):**
- Air temperature (degC)
- Relative humidity (%)
- Wind speed (m/s) and direction (deg)
- Sky/cloud cover (raw, station-dependent)
- Solar elevation & azimuth (deg) computed from latitude/longitude using `pvlib`

**Source:** NOAA/NCEI Climate Data Online (CDO) API — `global-hourly` (Integrated Surface Database).


In [1]:
# ===== 0) Install dependencies (fast) =====
!pip -q install pandas numpy requests pvlib python-dateutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 109.6 MB/s eta 0:00:00


In [2]:
# ===== 1) Imports =====
import os
import math
import time
import requests
import pandas as pd
import numpy as np
import pvlib

pd.set_option('display.max_columns', 200)

## 2) Configure (edit here)

You need a NOAA/NCEI CDO API token. Create one and paste it below.


In [4]:
# ===== 2) USER CONFIG =====
NOAA_TOKEN = "e97d47ec-be0b-11ee-84d9-0242ac130002-e97d48aa-be0b-11ee-84d9-0242ac130002"  # <-- required

# Site location (Rutgers New Brunswick default — edit if needed)
LAT = 40.4993
LON = -74.4474

# Date range (UTC). Use ISO format.
START_DATE = "2025-03-23"  # YYYY-MM-DD
END_DATE   = "2025-03-23"  # YYYY-MM-DD

# Output
OUT_CSV = "/content/noaa_env_hourly.csv"

# Optional: if you already know a station ID, set it (else auto-search)
STATION_ID = None  # e.g., "ISD:725050-14734" or None

assert NOAA_TOKEN and NOAA_TOKEN != "e97d47ec-be0b-11ee-84d9-0242ac130002-e97d48aa-be0b-11ee-84d9-0242ac130002"

AssertionError: 

## 3) Helper functions (CDO API)

We will:
1) Find a nearby `global-hourly` station (if not provided)  
2) Pull observations for the date range  
3) Pivot and standardize the fields  


In [ ]:
# ===== 3) CDO helpers =====
CDO_BASE = "https://www.ncei.noaa.gov/cdo-web/api/v2"
HEADERS = {"token": NOAA_TOKEN}

def cdo_get(endpoint: str, params: dict, max_retries: int = 5, sleep_s: float = 1.0):
    "GET with retries (handles rate limiting)."
    url = f"{CDO_BASE}/{endpoint}"
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if r.status_code == 429:
                time.sleep(sleep_s * attempt)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(sleep_s * attempt)
    raise RuntimeError(f"CDO request failed after retries: {last_err}")

def find_nearby_station(lat: float, lon: float, start_date: str, end_date: str, radius_deg: float = 0.75, limit: int = 25):
    "Find a nearby station in the global-hourly dataset using a bounding box search."
    extent = f"{lat + radius_deg},{lon - radius_deg},{lat - radius_deg},{lon + radius_deg}"
    params = {
        "datasetid": "global-hourly",
        "extent": extent,
        "startdate": start_date,
        "enddate": end_date,
        "limit": limit,
        "sortfield": "datacoverage",
        "sortorder": "desc",
    }
    data = cdo_get("stations", params)
    results = data.get("results", [])
    if not results:
        raise ValueError("No stations found. Increase radius_deg or use STATION_ID.")
    best = results[0]
    return best["id"], best

def fetch_hourly_data(station_id: str, start_date: str, end_date: str, datatypes=None, limit: int = 1000):
    "Pull global-hourly observations (handles pagination)."
    all_rows = []
    offset = 1
    while True:
        params = {
            "datasetid": "global-hourly",
            "stationid": station_id,
            "startdate": start_date,
            "enddate": end_date,
            "units": "metric",
            "limit": limit,
            "offset": offset
        }
        if datatypes:
            params["datatypeid"] = datatypes  # list ok
        data = cdo_get("data", params)
        rows = data.get("results", [])
        all_rows.extend(rows)

        meta = data.get("metadata", {})
        resultset = meta.get("resultset", {})
        count = resultset.get("count", 0)
        if offset + limit > count:
            break
        offset += limit

    return pd.DataFrame(all_rows)

def dewpoint_to_rh(temp_c: float, dewpoint_c: float) -> float:
    "Magnus formula RH percent from T and Td (degC)."
    if pd.isna(temp_c) or pd.isna(dewpoint_c):
        return np.nan
    a, b = 17.625, 243.04
    es = 6.1094 * math.exp(a * temp_c / (b + temp_c))
    e  = 6.1094 * math.exp(a * dewpoint_c / (b + dewpoint_c))
    rh = 100.0 * (e / es)
    return max(0.0, min(100.0, rh))

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

## 4) Pick station and pull NOAA hourly data

In [ ]:
# ===== 4) Station selection =====
if STATION_ID is None:
    station_id, station_meta = find_nearby_station(LAT, LON, START_DATE, END_DATE)
    print("Auto-selected station:", station_id)
    print("Station name:", station_meta.get("name"))
    print("Datacoverage:", station_meta.get("datacoverage"))
else:
    station_id = STATION_ID
    print("Using provided station:", station_id)

In [ ]:
# ===== 4B) Pull hourly data =====
df_raw = fetch_hourly_data(station_id, START_DATE, END_DATE, datatypes=None, limit=1000)
assert not df_raw.empty, "No data returned. Check station/date range/token."

print("Raw rows:", len(df_raw))
df_raw.head()

## 5) Pivot to wide format

CDO returns rows as (date, datatype, value). We pivot to get one row per timestamp.


In [ ]:
# ===== 5) Pivot =====
df_raw = df_raw.rename(columns={"date": "timestamp", "datatype": "datatype", "value": "value"})
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"], utc=True, errors="coerce")
df_raw = df_raw.dropna(subset=["timestamp"])

df_wide = df_raw.pivot_table(index="timestamp", columns="datatype", values="value", aggfunc="first").reset_index()
print("Wide columns:", len(df_wide.columns))
df_wide.head()

## 6) Standardize to the fields you need + solar angles

In [ ]:
# ===== 6) Build environmental table =====
temp_col = pick_col(df_wide, ["TMP", "TEMP", "AIR", "TAVG"])
dew_col  = pick_col(df_wide, ["DEW", "DEWP", "DPT"])
wspd_col = pick_col(df_wide, ["WND", "WSPD", "WSF2", "WSF", "AWND"])
wdir_col = pick_col(df_wide, ["WDR", "WDIR"])
sky_col  = pick_col(df_wide, ["SKY", "CLD", "CLOUD", "CIG", "CIGT"])

env = pd.DataFrame()
env["timestamp_utc"] = df_wide["timestamp"]

env["ambient_temp_C"] = df_wide[temp_col] if temp_col else np.nan
env["dewpoint_C"] = df_wide[dew_col] if dew_col else np.nan
env["wind_speed_mps"] = df_wide[wspd_col] if wspd_col else np.nan
env["wind_dir_deg"] = df_wide[wdir_col] if wdir_col else np.nan
env["sky_cover_raw"] = df_wide[sky_col] if sky_col else np.nan

env["relative_humidity_pct"] = [
    dewpoint_to_rh(t, d) for t, d in zip(env["ambient_temp_C"], env["dewpoint_C"])
]

# Solar position (UTC)
times = pd.DatetimeIndex(env["timestamp_utc"])
location = pvlib.location.Location(LAT, LON)
solpos = location.get_solarposition(times)

env["solar_elevation_deg"] = solpos["apparent_elevation"].values
env["solar_azimuth_deg"] = solpos["azimuth"].values

env = env.sort_values("timestamp_utc").reset_index(drop=True)
env.head()

## 7) Quality checks + export

In [ ]:
# ===== 7) QA + export =====
print("Selected datatype columns:",
      {"temp": temp_col, "dew": dew_col, "wspd": wspd_col, "wdir": wdir_col, "sky": sky_col})

print("Missing ambient_temp_C:", round(env["ambient_temp_C"].isna().mean(), 3))
print("Missing RH percent:", round(env["relative_humidity_pct"].isna().mean(), 3))
print("Missing wind_speed_mps:", round(env["wind_speed_mps"].isna().mean(), 3))

os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
env.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

In [ ]:
# Preview last 24 rows
env.tail(24)